In [8]:
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

# 1. Load the consolidated event registry
registry_path = Path("2026-05-07T11-14_export.csv")
df_events = pd.read_csv(registry_path)

date_col = 'Date'
penalty_col = 'Penalty (BPS)'

df_events[date_col] = pd.to_datetime(df_events[date_col]).dt.tz_localize(None)

# 2. Load VSTOXX from local CSV instead of yfinance
print("Loading local VSTOXX market data...")
vstoxx_path = Path("STOXX 50 Volatility VSTOXX EUR Historical Data.csv")
vstoxx_raw = pd.read_csv(vstoxx_path)

# Standardize VSTOXX dates
vstoxx_raw['Date'] = pd.to_datetime(vstoxx_raw['Date'])

# CRITICAL FIX: Investing.com exports are sorted newest-to-oldest!
# We MUST sort them oldest-to-newest before applying a time shift.
vstoxx_raw = vstoxx_raw.sort_values('Date', ascending=True).reset_index(drop=True)

# Isolate closing prices (named 'Price' in Investing.com CSV)
vstoxx_close = vstoxx_raw[['Date', 'Price']].copy()
vstoxx_close.columns = ['Merge_Date', 'VSTOXX']

# CRITICAL: Apply T-1 lag to prevent look-ahead bias
vstoxx_close['VSTOXX_T_minus_1'] = vstoxx_close['VSTOXX'].shift(1)
vstoxx_close['Merge_Date'] = vstoxx_close['Merge_Date'].dt.normalize()

# 3. Dataset alignment
df_events['Merge_Date'] = df_events[date_col].dt.normalize()

# Execute Left Join
df_merged = pd.merge(df_events, vstoxx_close[['Merge_Date', 'VSTOXX_T_minus_1']], 
                     on='Merge_Date', how='left')

# Handle weekend/holiday gaps via forward fill
df_merged['VSTOXX_T_minus_1'] = df_merged['VSTOXX_T_minus_1'].ffill()

# --- CRITICAL FIX: Removed the 'Detected == True' filter! ---
# We now use the full N=69 dataset to avoid selection bias.
# Zeros (No Trigger events) are explicitly kept in the regression.

# Drop any remaining NaNs
df_merged = df_merged.dropna(subset=[penalty_col, 'VSTOXX_T_minus_1'])

# 4. OLS Regression Setup (Penalty ~ VSTOXX)
Y = df_merged[penalty_col]
X = df_merged['VSTOXX_T_minus_1']
X = sm.add_constant(X) # Inject Alpha constant

# --- CRITICAL FIX: Added Heteroscedasticity-Consistent (HC3) standard errors ---
model = sm.OLS(Y, X).fit(cov_type='HC3')

# Output econometric summary
print("\n=== OLS REGRESSION RESULTS ===")
print(f"Total Observations analyzed: {len(df_merged)}")
print(model.summary())

Loading local VSTOXX market data...

=== OLS REGRESSION RESULTS ===
Total Observations analyzed: 69
                            OLS Regression Results                            
Dep. Variable:          Penalty (BPS)   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                 -0.007
Method:                 Least Squares   F-statistic:                    0.3006
Date:                Thu, 07 May 2026   Prob (F-statistic):              0.585
Time:                        16:03:29   Log-Likelihood:                -193.29
No. Observations:                  69   AIC:                             390.6
Df Residuals:                      67   BIC:                             395.0
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------